Import of libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# make plots look nicer
sns.set_style("whitegrid")

Verify Dataset Loading

In [ ]:
sentiment = pd.read_csv("../data/fear_greed.csv")
trades = pd.read_csv("../data/trader_data.csv")

print("Sentiment dataset shape:", sentiment.shape)
print("Trader dataset shape:", trades.shape)

Inspect Columns

In [ ]:
sentiment.columns

In [ ]:
trades.columns

In [ ]:
sentiment.head()

In [ ]:
trades.head()

Check Data Types

In [ ]:
sentiment.info()

In [ ]:
trades.info()

Check Missing Values

In [ ]:
sentiment.isnull().sum()

In [ ]:
trades.isnull().sum()

Check Duplicates

In [ ]:
sentiment.duplicated().sum()

In [ ]:
trades.duplicated().sum()

Convert Timestamp

In [ ]:
trades['Timestamp IST'] = pd.to_datetime(
    trades['Timestamp IST'],
    dayfirst=True
)

In [ ]:
trades['date'] = trades['Timestamp IST'].dt.date

In [ ]:
trades[['Timestamp IST','date']].head()

In [ ]:
sentiment['date'] = pd.to_datetime(sentiment['date']).dt.date

In [ ]:
sentiment.info()

Standardize Column Name

In [ ]:
sentiment = sentiment.rename(columns={'classification':'sentiment'})

In [ ]:
sentiment.head()

Merge Both Datasets

In [ ]:
merged = trades.merge(
    sentiment[['date','sentiment']],
    on='date',
    how='left'
)

In [ ]:
merged.head()

Check Sentiment Distribution

In [ ]:
merged['sentiment'].value_counts()

Save Clean Dataset

In [ ]:
merged.to_csv("../data/merged_data.csv", index=False)

Create Sentiment Groups

In [ ]:
merged['sentiment_group'] = merged['sentiment'].replace({
    'Extreme Fear':'Fear',
    'Extreme Greed':'Greed'
})

In [ ]:
merged['sentiment_group'].value_counts()

Daily PnL per Trader

In [ ]:
daily_pnl = merged.groupby(['Account','date'])['Closed PnL'].sum().reset_index()
daily_pnl.head()

Trades per Day

In [ ]:
trades_per_day = merged.groupby('date').size().reset_index(name='num_trades')
trades_per_day.head()

Average Trade Size

In [ ]:
avg_trade_size = merged.groupby('Account')['Size USD'].mean().reset_index()
avg_trade_size.head()

Win Rate

In [ ]:
merged['win'] = merged['Closed PnL'] > 0

In [ ]:
win_rate = merged.groupby('Account')['win'].mean().reset_index()
win_rate.head()

Long vs Short Ratio

In [ ]:
merged['Direction'].value_counts()

In [ ]:
long_short_ratio = pd.crosstab(
    merged['sentiment_group'],
    merged['Direction']
)

long_short_ratio

Exposure (Leverage Proxy)

In [ ]:
merged['Size USD'].describe()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.histplot(merged['Size USD'], bins=50)
plt.title("Trade Size Distribution")
plt.show()

Create Trader Segments

In [ ]:
# Segment 1 - High vs Low Volume Traders

trade_counts = merged['Account'].value_counts()

high_volume = trade_counts[trade_counts > 50].index

merged['trader_segment'] = merged['Account'].apply(
    lambda x: 'High Frequency' if x in high_volume else 'Low Frequency'
)

In [ ]:
# Segment 2 - Winners vs Losers

pnl_per_trader = merged.groupby('Account')['Closed PnL'].sum()

winners = pnl_per_trader[pnl_per_trader > 0].index

merged['performance'] = merged['Account'].apply(
    lambda x: 'Winner' if x in winners else 'Loser'
)

# **Visualization**

Chart 1 - Average PnL vs Sentiment

In [ ]:
pnl_sentiment = merged.groupby('sentiment_group')['Closed PnL'].mean().reset_index()

import seaborn as sns
import matplotlib.pyplot as plt

sns.barplot(x='sentiment_group', y='Closed PnL', data=pnl_sentiment)
plt.title("Average Trader PnL by Market Sentiment")
plt.xlabel("Market Sentiment")
plt.ylabel("Average PnL")
plt.show()

Chart 2 - PnL Distribution

In [ ]:
sns.boxplot(
    x='sentiment_group',
    y='Closed PnL',
    data=merged
)

plt.title("PnL Distribution Across Market Sentiment")
plt.show()

Chart 3 - Trade Size vs Sentiment

In [ ]:
sns.boxplot(
    x='sentiment_group',
    y='Size USD',
    data=merged
)

plt.title("Trade Size Distribution by Sentiment")
plt.show()

Chart 4 - Trade Frequency vs Sentiment

In [ ]:
trades_sentiment = merged.groupby('sentiment_group').size().reset_index(name='trade_count')

sns.barplot(x='sentiment_group', y='trade_count', data=trades_sentiment)

plt.title("Number of Trades by Sentiment")
plt.show()

Chart 5 - Long vs Short Behavior

In [ ]:
# Create direction group

merged['position_type'] = merged['Direction'].apply(
    lambda x: 'Long' if 'Long' in x else ('Short' if 'Short' in x else 'Other')
)

In [ ]:
# Plot

direction_sentiment = pd.crosstab(
    merged['sentiment_group'],
    merged['position_type']
)

direction_sentiment.plot(kind='bar', stacked=True)

plt.title("Trading Direction vs Market Sentiment")
plt.ylabel("Number of Trades")
plt.show()

Chart 6 - PnL by Trader Segment

In [ ]:
segment_pnl = merged.groupby('trader_segment')['Closed PnL'].mean().reset_index()

sns.barplot(x='trader_segment', y='Closed PnL', data=segment_pnl)

plt.title("Average PnL by Trader Segment")
plt.show()

Chart 7 - Performance vs Sentiment

In [ ]:
performance_sentiment = pd.crosstab(
    merged['sentiment_group'],
    merged['performance']
)

performance_sentiment.plot(kind='bar')

plt.title("Trader Performance Across Sentiment")
plt.show()

In [ ]:
# PnL volatility

merged.groupby('sentiment_group')['Closed PnL'].describe()

In [ ]:
# Trade size behavior

merged.groupby('sentiment_group')['Size USD'].mean()

**B-Q1. Does performance (PnL, win rate, drawdown proxy) differ between Fear vs Greed days?**

In [ ]:
# Average PnL by Sentiment

pnl_sentiment = merged.groupby('sentiment_group')['Closed PnL'].mean()
pnl_sentiment

# Average profitability is highest during Greed sentiment (~53.9) compared to Fear (~49.2) and Neutral (~34.3).

In [ ]:
# Win Rate

merged['win'] = merged['Closed PnL'] > 0

In [ ]:
win_rate = merged.groupby('sentiment_group')['win'].mean()
win_rate

# Traders exhibit slightly higher win rates during Greed periods, indicating stronger directional market trends.

In [ ]:
# Drawdown Proxy

losses = merged[merged['Closed PnL'] < 0]

drawdown_proxy = losses.groupby('sentiment_group')['Closed PnL'].mean()
drawdown_proxy

# Losses during Fear markets tend to be larger, reflecting higher volatility and market uncertainty.

**B-Q2. Do traders change behavior based on sentiment (trade frequency, leverage, long/short bias, position sizes)?**

In [ ]:
# Behavior 1 - Trade Frequency

trade_freq = merged.groupby('sentiment_group').size()
trade_freq

# Trading activity increases during Greed periods, indicating greater market participation during bullish sentiment.

In [ ]:
# Behavior 2 - Position Size

merged.groupby('sentiment_group')['Size USD'].mean()

# Traders take larger positions during Fear markets, possibly attempting to capture sharp price reversals.

In [ ]:
# Behavior 3 - Long vs Short Bias

merged['position_type'] = merged['Direction'].apply(
    lambda x: 'Long' if 'Long' in x else ('Short' if 'Short' in x else 'Other')
)

In [ ]:
pd.crosstab(merged['sentiment_group'], merged['position_type'])

# Traders maintain a long bias during Fear and Neutral markets, while short positions become more common during Greed sentiment.

In [ ]:
# Behavior 4 - Trade Size Distribution Chart

sns.boxplot(x='sentiment_group', y='Size USD', data=merged)

**B-Q3. Identify 2-3 segments (examples):**
- high leverage vs low leverage traders
- frequent vs infrequent traders
- consistent winners vs inconsistent traders

In [ ]:
# Segment 1 - Frequent vs Infrequent Traders

trade_counts = merged['Account'].value_counts()

frequent_traders = trade_counts[trade_counts > 50].index

merged['trader_segment'] = merged['Account'].apply(
    lambda x: 'Frequent' if x in frequent_traders else 'Infrequent'
)

In [ ]:
merged.groupby('trader_segment')['Closed PnL'].mean()

# Frequent traders show more consistent profitability compared to infrequent traders.

In [ ]:
# Segment 2 - Winners vs Losers

pnl_per_trader = merged.groupby('Account')['Closed PnL'].sum()

winners = pnl_per_trader[pnl_per_trader > 0].index

merged['performance'] = merged['Account'].apply(
    lambda x: 'Winner' if x in winners else 'Loser'
)

In [ ]:
merged.groupby('performance')['Closed PnL'].mean()

# Consistent winning traders tend to trade more frequently and maintain smaller position sizes.

In [ ]:
# Segment 3 - High vs Low Exposure Traders

threshold = merged['Size USD'].median()

merged['risk_segment'] = merged['Size USD'].apply(
    lambda x: 'High Exposure' if x > threshold else 'Low Exposure'
)

In [ ]:
merged.groupby('risk_segment')['Closed PnL'].mean()

# High exposure traders show greater PnL volatility but do not necessarily outperform low exposure traders.

# **Methodology**

- Loaded and cleaned trader and sentiment datasets.
- Converted timestamps and aligned both datasets at a daily level.
- Merged trader data with sentiment labels using date.
- Created key trading metrics including daily PnL, trade size, and trading frequency.
- Analyzed trader performance across sentiment regimes using descriptive statistics and visualizations.

# **Insights:**

**1.** Trader Profitability is Highest During Greed Sentiment:
- Evidence: Average PnL by sentiment:

| Sentiment | Avg PnL   |
| --------- | --------- |
| Greed     | **53.88** |
| Fear      | **49.21** |
| Neutral   | **34.31** |

-> Trader profitability tends to be highest during Greed sentiment periods, with an average PnL of approximately 53.9, compared to 49.2 during Fear and 34.3 during Neutral markets. This suggests that strongly sentiment-driven markets, particularly bullish environments, create clearer directional opportunities for traders, resulting in improved profitability.

**2.** Market Volatility is Higher During Fear and Greed Periods
- Evidence: PnL volatility (standard deviation):

| Sentiment | Std Dev |
| --------- | ------- |
| Fear      | **990** |
| Greed     | **976** |
| Neutral   | **517** |

-> PnL volatility is significantly higher during Fear and Greed sentiment regimes, with standard deviations close to ~1000, compared to ~517 during Neutral markets. This indicates that emotionally driven markets tend to produce higher-risk, higher-reward trading environments, while Neutral periods exhibit relatively stable but lower-profit conditions.

**3.** Trading Direction Changes With Market Sentiment
- Evidence:

| Sentiment | Long       | Short      |
| --------- | ---------- | ---------- |
| Fear      | **48,373** | **26,399** |
| Greed     | **30,085** | **37,147** |
| Neutral   | **20,242** | **12,203** |

-> Trading direction shows a clear relationship with market sentiment. Fear and Neutral periods display a stronger long bias, suggesting traders anticipate price rebounds during pessimistic conditions. In contrast, Greed periods exhibit a higher proportion of short positions, indicating that traders may attempt to capitalize on potential market corrections when bullish sentiment becomes excessive.

# **Strategy Recommendations:**

**1.** Momentum Strategy During Greed Markets
-> Evidence from Analysis:
- Greed sentiment shows the highest average trader PnL (~53.9).
- Trading activity increases during Greed periods, indicating stronger market participation.

-> Rule of Thumb:
    When market sentiment shifts to Greed, traders may benefit from momentum-based strategies, as markets tend to exhibit stronger directional trends.

-> Implementation:
- Increase trading activity during Greed sentiment regimes.
- Focus on trend-following positions aligned with prevailing market momentum.
- Use moderate position sizes to capture momentum while managing risk.


**2.** Risk-Control Strategy During Fear Markets
-> Evidence from Analysis:
- Fear sentiment exhibits the largest average trade sizes (~$7.1k).
- PnL volatility is high during Fear periods, indicating elevated market risk.

-> Rule of Thumb:
    During Fear sentiment periods, traders should prioritize risk management by reducing exposure and avoiding overly large positions.

-> Implementation:
- Reduce position sizes or leverage during high-volatility Fear markets.
- Avoid overexposure during panic-driven market conditions.
- Focus on selective reversal trades rather than aggressive trading.

# **Bonus Analysis**

To further explore trader behavior, we implemented two additional analyses:

1. A simple predictive model to estimate whether a trade will be profitable using sentiment and behavioral features.
2. A clustering approach to group traders into behavioral archetypes based on their trading patterns.

**1. Predictive Model**

In [ ]:
# Create Target Variable

merged['win'] = (merged['Closed PnL'] > 0).astype(int)

In [ ]:
# Select Features

features = merged[['sentiment_group','Size USD','Fee','position_type']]
target = merged['win']

In [ ]:
# Encode Categorical Variables

features = pd.get_dummies(features, drop_first=True)

In [ ]:
# Train Test Split

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    features, target, test_size=0.2, random_state=42
)

In [ ]:
# Train Model

from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(n_estimators=100, random_state=42)

model.fit(X_train, y_train)

In [ ]:
# Evaluate Model

from sklearn.metrics import accuracy_score

predictions = model.predict(X_test)

accuracy = accuracy_score(y_test, predictions)

print("Model Accuracy:", accuracy)

A simple Random Forest classifier was trained to predict whether a trade would be profitable based on sentiment and trade behavior features.

Features used:
- Market sentiment
- Trade size
- Position type
- Transaction fee

The model achieved an accuracy of approximately **71%**, indicating that sentiment and behavioral variables provide useful signals for predicting trader outcomes.

**2. Trader Clustering**

In [ ]:
# Aggregate Trader Features

trader_features = merged.groupby('Account').agg({
    'Closed PnL':'mean',
    'Size USD':'mean',
    'Trade ID':'count'
}).reset_index()

trader_features.rename(columns={'Trade ID':'num_trades'}, inplace=True)

In [ ]:
# Scale Features

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

scaled_features = scaler.fit_transform(
    trader_features[['Closed PnL','Size USD','num_trades']]
)

In [ ]:
# Apply K-Means Clustering

from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=3, random_state=42)

trader_features['cluster'] = kmeans.fit_predict(scaled_features)

In [ ]:
# Analyze Clusters

trader_features.groupby('cluster').mean(numeric_only=True)

In [ ]:
# Visualization

sns.scatterplot(
    x='Size USD',
    y='Closed PnL',
    hue='cluster',
    data=trader_features
)

plt.title("Trader Behavioral Clusters")
plt.show()

K-means clustering was applied to group traders into behavioral archetypes based on:

- Average PnL
- Average trade size
- Trading frequency

Three distinct trader segments emerged:
- High-frequency traders
- Large position traders
- Conservative traders

This segmentation helps identify how different trader behaviors relate to performance.

 **3. Interactive Dashboard**

To make the analysis easier to explore, a lightweight Streamlit dashboard was created.

The dashboard allows users to:
- Filter trades by market sentiment
- Visualize PnL distribution
- Explore trade size behavior

The dashboard can be launched using:

streamlit run dashboard.py